# Week 5 — E2: Quantum VQC+GAT Full Dataset (Laptop)

Runs ~37 hours continuously. Auto-saves every epoch to `E2_quantum_latest.pth`. Safe to interrupt and resume.

In [1]:
import os, sys, time, json, gc
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import roc_auc_score, f1_score

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = \
    'expandable_segments:True,max_split_size_mb:512'

sys.path.insert(0, '..')
from pathq.model_v2   import QuantaPathV2
from pathq.dataset_v2 import get_loaders_from_features
from torch_geometric.loader import DataLoader as PyGLoader

DEVICE   = torch.device('cuda')
ROOT     = Path('..')
FEAT_DIR = Path('./data/features_uni')   # notebooks/data/features_uni
CKPT_DIR = ROOT / 'checkpoints'
OUT_DIR  = ROOT / 'outputs'
CKPT_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

torch.manual_seed(42)
np.random.seed(42)

print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')
print(f'Feat : {len(list(FEAT_DIR.glob("*.pt")))} slides')

/home/kabi/.conda/envs/pathq/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[model_v2] Using Transformer for global branch
GPU  : NVIDIA GeForce RTX 5060 Laptop GPU
VRAM : 8.1GB
Feat : 333 slides


In [2]:
train_loader, val_loader, test_loader = get_loaders_from_features(
    features_dir = FEAT_DIR,
    batch_size   = 4,
    k            = 8,
    seed         = 42,
    max_patches  = 2048,
)
print(f'Train: {len(train_loader)} batches')
print(f'Val:   {len(val_loader)} batches')
print(f'Test:  {len(test_loader)} batches')

Split: train=233 (pos=78) val=50 (pos=17) test=50 (pos=16)
Train: 59 batches
Val:   13 batches
Test:  13 batches


In [3]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss, n = 0.0, 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        torch.cuda.empty_cache()
        logits, _ = model(batch)
        loss      = F.cross_entropy(logits, batch.y.view(-1))
        loss_val  = loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        del logits, loss
        torch.cuda.empty_cache()
        total_loss += loss_val
        n += 1
    return total_loss / max(n, 1)

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    probs, labels, tl, n = [], [], 0.0, 0
    for batch in loader:
        batch     = batch.to(device)
        logits, _ = model(batch)
        tl       += F.cross_entropy(logits, batch.y.view(-1)).item()
        probs.extend(torch.softmax(logits, 1)[:, 1].cpu().tolist())
        labels.extend(batch.y.view(-1).cpu().tolist())
        n += 1
    p, l  = np.array(probs), np.array(labels)
    preds = (p >= 0.5).astype(int)
    auc   = roc_auc_score(l, p) if len(np.unique(l)) > 1 else 0.5
    f1    = f1_score(l, preds, zero_division=0)
    tp=int(((preds==1)&(l==1)).sum())
    fn=int(((preds==0)&(l==1)).sum())
    tn=int(((preds==0)&(l==0)).sum())
    fp=int(((preds==1)&(l==0)).sum())
    return {
        'auc':         round(auc, 6),
        'f1':          round(f1,  6),
        'loss':        round(tl/max(n,1), 6),
        'sensitivity': round(tp/max(tp+fn,1), 4),
        'specificity': round(tn/max(tn+fp,1), 4),
    }

SEP  = '═' * 65
DASH = '─' * 65

def run_E2(model, tr, va, te, device,
           ckpt_best, ckpt_latest,
           epochs=30, lr=3e-5, patience=5):

    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-3
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=1e-6
    )
    best_val_auc = 0.0
    patience_ctr = 0
    start_ep     = 1

    # Auto-resume from latest checkpoint
    if Path(ckpt_latest).exists():
        ck = torch.load(ckpt_latest, weights_only=False)
        model.load_state_dict(ck['model_state'])
        start_ep     = ck['epoch'] + 1
        best_val_auc = ck['best_val_auc']
        # Restore scheduler state
        for _ in range(start_ep - 1):
            scheduler.step()
        print(f'Resumed from epoch {start_ep-1}, '
              f'best_val_auc={best_val_auc:.4f}')

    print(f'\n{SEP}')
    print(f' E2 \u2014 Quantum VQC+GAT | Full CAMELYON16 | 3000 patches')
    print(f' n_qubits=3, n_layers=2, lr={lr}, patience={patience}')
    print(f'{SEP}')
    print(f' {"Ep":>3}  {"TrLoss":>8}  {"VaLoss":>8}  '
          f'{"VaAUC":>7}  {"VaF1":>7}  {"Secs":>6}')
    print(f'{DASH}')

    for ep in range(start_ep, epochs + 1):
        t0         = time.time()
        train_loss = train_one_epoch(model, tr, optimizer, device)
        val_m      = evaluate(model, va, device)
        scheduler.step()
        flag = ''

        if val_m['auc'] > best_val_auc:
            best_val_auc = val_m['auc']
            patience_ctr = 0
            flag         = '\u2713'
            torch.save({
                'model_state':  model.state_dict(),
                'epoch':        ep,
                'best_val_auc': best_val_auc,
            }, ckpt_best)
        else:
            patience_ctr += 1

        # Save every epoch for safe resume
        torch.save({
            'model_state':  model.state_dict(),
            'epoch':        ep,
            'best_val_auc': best_val_auc,
        }, ckpt_latest)

        overfit = ' \u26a0 overfit' if val_m['loss'] > train_loss * 2.5 else ''
        print(f' {ep:>3}  {train_loss:>8.4f}  {val_m["loss"]:>8.4f}  '
              f'{val_m["auc"]:>7.4f}  {val_m["f1"]:>7.4f}  '
              f'{int(time.time()-t0):>5}s  {flag}{overfit}')

        if patience_ctr >= patience:
            print(f'\n Early stop at epoch {ep}')
            break

        torch.cuda.empty_cache()
        gc.collect()

    ck = torch.load(ckpt_best, weights_only=False)
    model.load_state_dict(ck['model_state'])
    test_m = evaluate(model, te, device)

    print(f'{DASH}')
    print(f' Best val AUC : {best_val_auc:.4f}')
    print(f' Test AUC     : {test_m["auc"]:.4f}')
    print(f' F1           : {test_m["f1"]:.4f}')
    print(f' Sensitivity  : {test_m["sensitivity"]:.4f}')
    print(f' Specificity  : {test_m["specificity"]:.4f}')
    print(f' Overfit gap  : {test_m["auc"]-best_val_auc:+.4f}')
    print(f'{SEP}')
    return {**test_m, 'val_auc': best_val_auc,
            'gap': round(test_m['auc']-best_val_auc, 6)}

print('\u2705 Training functions ready')

✅ Training functions ready


In [4]:
CKPT_BEST   = str(CKPT_DIR / 'E2_quantum_best.pth')
CKPT_LATEST = str(CKPT_DIR / 'E2_quantum_latest.pth')

model_E2 = QuantaPathV2(
    use_vqc    = True,
    n_qubits   = 3,
    vqc_layers = 2,
    in_dim     = 1040,
).to(DEVICE)

n_params = sum(p.numel() for p in model_E2.parameters()
               if p.requires_grad)
print(f'Trainable params: {n_params:,}')
print('Starting E2 \u2014 estimated ~37 hours')
print('Auto-saves every epoch to E2_quantum_latest.pth')
print('Resumes automatically if interrupted')
print()

result_E2 = run_E2(
    model_E2,
    train_loader, val_loader, test_loader,
    DEVICE,
    ckpt_best   = CKPT_BEST,
    ckpt_latest = CKPT_LATEST,
    epochs      = 30,
    lr          = 3e-5,
    patience    = 5,
)

with open(OUT_DIR / 'E2_result.json', 'w') as f:
    json.dump({'experiment': 'E2_quantum', **result_E2}, f, indent=2)

print('\u2705 E2 complete')
print(f'Checkpoint: {CKPT_BEST}')
print('Back up E2_quantum_best.pth immediately')

[VQC] lightning.gpu (3q, 2L) — adjoint gradients
QuantaPathV2: use_vqc=True, trainable=970,385
Trainable params: 970,385
Starting E2 — estimated ~37 hours
Auto-saves every epoch to E2_quantum_latest.pth
Resumes automatically if interrupted


═════════════════════════════════════════════════════════════════
 E2 — Quantum VQC+GAT | Full CAMELYON16 | 3000 patches
 n_qubits=3, n_layers=2, lr=3e-05, patience=5
═════════════════════════════════════════════════════════════════
  Ep    TrLoss    VaLoss    VaAUC     VaF1    Secs
─────────────────────────────────────────────────────────────────
   1    0.6817    0.6803   0.5383   0.4103   5797s  ✓


KeyboardInterrupt: 